# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible guide for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Title: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets and fields. All entities are referenced and operated on by their Croissant `@id`.

Let's enumerate the record sets, their fields, and each field's `@id`.

In [ ]:
#--- Utility to pretty-print summary
import json

# The metadata.record_sets() function lists all record sets
record_sets = list(dataset.record_sets())

print(f"# Number of Record Sets: {len(record_sets)}\n")
for recset in record_sets:
    print(f"Record Set Name: {getattr(recset, 'name', None)}")
    print(f"\t@id: {getattr(recset, '@id', None)}")
    print(f"\tDescription: {getattr(recset, 'description', None)}")
    print("\tFields:")
    for field in recset.fields:
        print(f"\t    Field name: {getattr(field, 'name', None)}; @id: {getattr(field, '@id', None)}; dataType: {getattr(field, 'data_type', None)}")
    print()

#### Example: Preview the first few records of a record set

Let's choose a record set `@id` from above to preview. You may replace the record set ID below with any available from the output above.

In [ ]:
# Preview a small sample from the first record set

if record_sets:
    record_set_id = getattr(record_sets[0], '@id')
    print(f"Previewing records for record set @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i > 2:
            break

## 3. Data Extraction
For further analysis, load the data from each record set into DataFrames using their `@id`.

In [ ]:
# Build a dictionary of DataFrames keyed by record set @id
dataframes = {}
record_set_ids = [getattr(rs, '@id') for rs in record_sets]

for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)

# Print available columns for each loaded record set
for rec_id in record_set_ids:
    print(f"Record set @id: {rec_id}")
    if not dataframes[rec_id].empty:
        print(f"  Columns: {dataframes[rec_id].columns.tolist()}")
        display(dataframes[rec_id].head(3))
    else:
        print("  [No records]")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. Here, all references to columns (fields) use the corresponding Croissant `@id` of the field, as shown above.

In [ ]:
# Example: Filter, normalize, and group data using @id references
import numpy as np

# For demonstration, pick the first record set with at least one numeric field
selected_rec_id = None
numeric_field_id = None
group_field_id = None
for recset in record_sets:
    df = dataframes[getattr(recset, '@id')]
    # Find a numeric field
    for field in recset.fields:
        # Accept integer or float fields
        if getattr(field, 'data_type', None) in ('schema:Float', 'schema:Integer', 'Number', 'Float', 'Integer') and getattr(field, '@id', None) in df:
            selected_rec_id = getattr(recset, '@id')
            numeric_field_id = getattr(field, '@id')
            break
    # Heuristically, try to pick a group field that's non-numeric
    if selected_rec_id is not None:
        for field in recset.fields:
            if getattr(field, 'data_type', None) in ('schema:Text', 'Text', None) and getattr(field, '@id', None) in df:
                group_field_id = getattr(field, '@id')
                break
        break

if selected_rec_id and numeric_field_id:
    df = dataframes[selected_rec_id].copy()
    # Coerce to numeric, errors='coerce' makes invalids NaN
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.9 if len(df) > 20 else 0.5)
    filtered_df = df[df[numeric_field_id] > threshold].dropna(subset=[numeric_field_id])
    print(f"Filtered records in @{selected_rec_id} with {numeric_field_id} > {threshold:.2f} (90th percentile): {len(filtered_df)} records\n")
    display(filtered_df.head())
    
    # Normalization
    col_norm = numeric_field_id + "_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nSample normalized values for column @{numeric_field_id}:")
    display(filtered_df[[numeric_field_id, col_norm]].head())
    
    # Group by group_field if available
    if group_field_id and group_field_id in filtered_df:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by @{group_field_id}:")
        display(grouped.head())
else:
    print('Dataset does not contain a numeric field in the detected record sets.')

## 5. Visualization
Let's visualize the distribution of a numeric field and, optionally, the mean values across groups using Python plotting libraries.

In [ ]:
# Visualize the distribution and group means if data is available
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rec_id and numeric_field_id:
    # Histogram
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(data=dataframes[selected_rec_id], x=numeric_field_id, bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of @{numeric_field_id}")
    plt.show()

    # Grouped bar chart if group_field_id available
    if group_field_id and group_field_id in filtered_df:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
        plt.title(f"Mean @{numeric_field_id} by @{group_field_id} (filtered records)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you learned how to:
- Load a Croissant schema-defined dataset using `mlcroissant`;
- Inspect the dataset structure by record set and field `@id`;
- Extract and process data with all entity references by `@id`;
- Apply filtering, normalization, and grouped analysis by field `@id`s;
- Visualize numeric field distributions and group means.

With this workflow, you can explore FAIR scientific datasets in a way that's robust and reproducible, while maintaining full referential transparency via Croissant `@id` fields.